# Course 26 — Enterprise Architecture Governance

This notebook executes the trusted Course 26 reference, then guides analysis of its decisions and evidence limits. It does not authorize a release.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
for module_name in list(sys.modules):
    if module_name == 'governance_app' or module_name.startswith('governance_app.'):
        del sys.modules[module_name]
sys.path.insert(0, str(ROOT))
from governance_app.control_plane import ArchitectureControlPlane
from governance_app.simulation import run_reference

result = run_reference(ROOT)
assert result['golden_passed']
assert result['adversarial_evaluation']['metrics']['exact_decision_accuracy'] == 1.0
result['policy_version'], result['stale_report_result']

## Compare decision classes

Notice the difference between a known violation (`BLOCKED`), an architectural choice that needs accountable review (`REVIEW_REQUIRED`), and a runtime destination that static analysis cannot establish (`INSUFFICIENT_EVIDENCE`).

In [ ]:
decision_table = [(item['scenario_id'], item['actual'], item['finding_codes']) for item in result['golden_scenarios']]
assert len(decision_table) == 13
decision_table

## Inspect targeted invalidation

A governing decision should invalidate registered dependents, not every repository in the enterprise.

In [ ]:
impact = ArchitectureControlPlane(ROOT).impact('ADR-028-RETRIEVAL-PLATFORM')
assert impact['decision'] == 'TARGETED_REASSESSMENT_REQUIRED'
assert 'ai-service' in impact['repositories']
impact

## Reflection

For one architecture rule in your organization, identify its owner, exact scope, trusted fact producer, uncertainty state, failure disposition, exception authority, compensating evidence, and change-impact path.